In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, FunctionTransformer
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, log_loss

In [ ]:
df = pd.read_csv('train.csv')
df

# 1. EDA

In [ ]:
df.info()       # tonns of missing values

In [ ]:
df.drop('id', axis=1).duplicated().sum()    # exist 6 duplicated rows

In [ ]:
df.drop_duplicates(subset=df.columns.difference(['id']), inplace=True)      # dropping duplicates

In [ ]:
df.describe()

`Age` column is in days

`N_Days` represents how long the patient was observed until the outcome occurred. This information is not available at the time of prediction, because clinicians cannot wait for several days to see what happens before making a prediction. Additionally, the class **C (censored)** only means the patient was alive at last follow-up, not that they will remain alive indefinitely. Therefore, `N_Days` should be removed from the model to ensure predictions are based only on information available at baseline.


In [ ]:
df.Status.value_counts()       # target feature is highly imbalanced, and has one wrong value

In [ ]:
df = df[df.Status != 'Y']

In [ ]:
survival_rate = df['Status'].value_counts() / len(df) * 100             # 67/30 majority/priority rate

plt.figure(figsize=(5,5))
plt.pie(survival_rate, labels = ['C', 'D', 'CL'], autopct='%1.1f%%')
plt.title('Distribution of patient status')
plt.show()

In [ ]:
cols = ['Age', 'Bilirubin', 'Cholesterol', 'Copper', 'Alk_Phos',
        'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Outliers', fontsize=16)

for ax, col in zip(axes.flatten(), cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()


Outliers are to be capped inside pipeline during training

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(15,10))
fig.suptitle('Relationship of Status with other indicators', fontsize=16)

sns.countplot(x='Sex', hue='Status', palette='viridis', data=df, ax=axes[0,0])
axes[0,0].set_title('Gender')

sns.countplot(x='Stage', hue='Status', palette='viridis', data=df, ax=axes[0,1])
axes[0,1].set_title('Stage')

bins=[0, 30, 40, 50, 60, 120]
labels = ["0-30", "31-40", "41-50", '51-60', "61+"]
sns.countplot(x=pd.cut(df['Age']/365, bins=bins, labels=labels), hue=df['Status'], palette='viridis', ax=axes[1,0])
axes[1,0].set_title('Age intervals')

sns.countplot(x='Edema', hue='Status', palette='viridis', data=df, ax=axes[1,1])
axes[1,1].set_title('Edema')

plt.tight_layout()
plt.show()

- Higher disease stage and older age are associated with a higher risk of death
- The dataset is heavily imbalanced with respect to gender

# 2. Model

In [ ]:
X = df.drop('Status', axis=1)

le = LabelEncoder()
y = le.fit_transform(df.Status.copy())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

The dataset contains many missing and abnormal values (e.g., extreme age or lab measurements), but the test set also includes such cases. LightGBM is used because it handles missing values internally and can still make predictions on incomplete or noisy data. If higher-quality data becomes available, the model can be retrained to improve performance.

In [ ]:
def cap_outliers(x):
    """Cap outliers"""

    x['Age'] = x['Age'].clip(upper=43_800)                  # 120*365=43800 days

    x['Bilirubin'] = x['Bilirubin'].clip(upper=20)     

    x['Cholesterol'] = x['Cholesterol'].clip(upper=700)

    x['Copper'] = x['Copper'].clip(upper=300)

    x['Alk_Phos'] = x['Alk_Phos'].clip(upper=8000)

    x['SGOT'] = x['SGOT'].clip(upper=500)

    x['Tryglicerides'] = x['Tryglicerides'].clip(upper=400)

    x['Platelets'] = x['Platelets'].clip(upper=600)

    return x.drop(['N_Days'], axis=1)

cap_features = FunctionTransformer(cap_outliers, validate=False)

categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ], remainder='passthrough'      # pass numerical features to pipeline
    )

In [ ]:
print(le.classes_)

In [ ]:
weights = {0: 1, 1: 1, 2: 67/30}        # ratio of majority/priority class for class_weight to address class imbalance

In [ ]:
# LGBM handles missing values

pipeline = Pipeline([
    ('cap_outliers', cap_features),
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        objective='multiclass',
        num_class=3,
        class_weight=weights
        ))
])

In [ ]:
pipeline.fit(X_train, y_train)                  # training model

# 3. Model evaluation

In [ ]:
print(le.classes_)

In [ ]:
y_pred = pipeline.predict(X_test)
labels = ['C','CL','D']
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, xticklabels=labels, yticklabels=labels, fmt='d', annot=True)
plt.title('Confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

The model produced 176 false negatives, meaning 176 patients who died were incorrectly predicted as alive.

In [ ]:
y_pred_proba = pipeline.predict_proba(X_test)

In [ ]:
loss = log_loss(y_test, y_pred_proba)
confidence = np.exp(-0.42)

print(f'Logarithmic loss: {loss:.4f}')
print(f'Confidence: {confidence:.4f}')

The average predicted probability for the true class is approximately 0.66, indicating moderate confidence.

# 4. Test

In [ ]:
df_test = pd.read_csv('test.csv')
df_test

In [ ]:
survival_rate = pipeline.predict_proba(df_test)

In [ ]:
df_predict = pd.read_csv('sample_submission.csv')
df_predict

In [ ]:
df_predict['Status_C'] = survival_rate[:,0]
df_predict['Status_CL'] = survival_rate[:,1]
df_predict['Status_D'] = survival_rate[:,2]
df_predict

In [ ]:
df_predict.to_csv('cirrhosis_survival_rate.csv', index=False)